# Лабораторная работа №1
# Сценарное и функциональное программирование

Последовательный запуск: конфигурация → геолокация → прогноз → сохранение в БД → экспорт отчёта.

Запустите ячейки сверху вниз. Перед запуском проверьте `.env` и доступность базы данных.

In [3]:
from pathlib import Path
import os
from dotenv import load_dotenv
from sqlalchemy.exc import SQLAlchemyError

from geo_locator import get_location, GeoLocationError
from weather_client import get_forecast, WeatherAPIError
from db_models import get_engine, init_db, save_forecast, get_city_forecast
from exporter import export_markdown

BASE_DIR = Path.cwd()
print(f"Директория проекта: {BASE_DIR}")

Директория проекта: c:\Users\yeaok\scenarnoe\laba1


## 1. Загрузка конфигурации

In [ ]:
env_path = BASE_DIR / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Не найден файл конфигурации: {env_path}")

load_dotenv(env_path, override=True)
if not os.getenv("WEATHER_API_KEY"):
    raise ValueError("Не задан WEATHER_API_KEY в .env")

print("WEATHER_API_KEY успешно получен.")

WEATHER_API_KEY успешно получен


## 2. Определение города и координат

In [6]:
location = get_location()
city = location.city
latitude = location.latitude
longitude = location.longitude

print(f"Город: {city}")
print(f"Координаты: {latitude:.4f}, {longitude:.4f}")

Город: Moscow
Координаты: 55.7558, 37.6173


## 3. Получение прогноза погоды

In [8]:
forecast = get_forecast(latitude, longitude)
if not forecast:
    raise ValueError("API не вернул данные прогноза")

print(f"Получено записей прогноза: {len(forecast)}")
forecast[:4]

Получено записей прогноза: 4


[{'forecast_date': datetime.date(2026, 9, 29),
  'temp_min': 9.22,
  'temp_max': 15.43,
  'humidity': 64,
  'wind_speed': 1.62,
  'description': 'ясно'},
 {'forecast_date': datetime.date(2026, 9, 30),
  'temp_min': 8.98,
  'temp_max': 15.94,
  'humidity': 78,
  'wind_speed': 2.99,
  'description': 'ясно'},
 {'forecast_date': datetime.date(2026, 10, 1),
  'temp_min': 7.02,
  'temp_max': 12.37,
  'humidity': 79,
  'wind_speed': 2.98,
  'description': 'пасмурно'},
 {'forecast_date': datetime.date(2026, 10, 2),
  'temp_min': 5.17,
  'temp_max': 11.88,
  'humidity': 78,
  'wind_speed': 3.71,
  'description': 'пасмурно'}]

## 4. Определение дат прогноза

In [9]:
dates = sorted({str(item["forecast_date"]) for item in forecast})
print("Даты прогноза:")
for forecast_date in dates:
    print(f" - {forecast_date}")

Даты прогноза:
 - 2026-09-29
 - 2026-09-30
 - 2026-10-01
 - 2026-10-02


## 5. Сохранение данных в базу данных

In [10]:
engine = get_engine()
init_db(engine)
saved_count = save_forecast(engine, city, forecast)
records = get_city_forecast(engine, city)

print(f"Новых записей сохранено: {saved_count}")
print(f"Всего записей города в БД: {len(records)}")

Новых записей сохранено: 4
Всего записей города в БД: 4


## 6. Экспорт результатов в Markdown

In [11]:
output_file = os.getenv("OUTPUT_FILE", "weather_report.md")
output_path = Path(output_file)
if not output_path.is_absolute():
    output_path = BASE_DIR / output_path
output_path.parent.mkdir(parents=True, exist_ok=True)

export_markdown(records, str(output_path))
print(f"Отчёт сохранён: {output_path.resolve()}")

Отчёт сохранён: C:\Users\yeaok\scenarnoe\laba1\weather_report.md


## 7. Итоговый отчёт

In [14]:
date_range = f"{dates[0]} — {dates[-1]}" if dates else "нет данных"
print("")
print("             Лабораторная работа №1 - Сценарное и функциональное программирование")
print("")
print(f"Город: {city}")
print(f"Координаты: {latitude:.4f}, {longitude:.4f}")
print(f"Даты прогноза: {date_range}")
print(f"Количество дат: {len(dates)}")
print(f"Новых записей сохранено: {saved_count}")
print(f"Всего записей города в БД: {len(records)}")
print(f"Выходной файл: {output_path.resolve()}")
print("")
print("Выполнение было успешно завершёно.")


             Лабораторная работа №1 - Сценарное и функциональное программирование

Город: Moscow
Координаты: 55.7558, 37.6173
Даты прогноза: 2026-09-29 — 2026-10-02
Количество дат: 4
Новых записей сохранено: 4
Всего записей города в БД: 4
Выходной файл: C:\Users\yeaok\scenarnoe\laba1\weather_report.md

Выполнение было успешно завершёно.
